 # **Lab 04 – Spark Streaming**

## **Group: CDHD**

### **Team Members**

| STT | Full Name           | Student ID |
| :-: | ------------------- | :--------: |
|  1  | Phan Thị Phương Chi |  23120025  |
|  2  | Trần Thanh Đạt      |  23120030  |
|  3  | Phạm Ngọc Duy       |  23120035  |
|  4  | Lê Hoàng Mỹ Hạ      |  23120038  |

## **Task 1: Repository Cloning and File Discovery**

### Mục tiêu
- Khám phá toàn bộ kho mã nguồn Python được chọn làm **corpus** cho CPG Parser Service (Task 2).
- Kết quả của task này là danh sách file `.py` đã phân loại, lưu vào `output/discovered_files.csv`. 
  
### Thông tin repo

| Trường        | Giá trị |
|:--------------|:--------|
| **Tên repo**  | `peft` |
| **Tổ chức**   | `huggingface` |
| **URL clone** | `https://github.com/huggingface/peft.git` |

### 0. Imports & cấu hình đường dẫn

In [1]:
import os
import sys
import subprocess
import platform
from pathlib import Path

import pandas as pd

# Project root = thư mục chứa notebook này (spark-streaming-lab)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "peft").exists() and (PROJECT_ROOT.parent / "peft").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Đường dẫn repo peft & output trên máy hiện tại (trong project)
WIN_REPO_PATH = str(PROJECT_ROOT / "peft")
WIN_OUTPUT_DIR = str(PROJECT_ROOT / "output")

# Đường dẫn tương đương khi chạy trong WSL (Windows drive C: -> /mnt/c/...)
# C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\peft
WSL_REPO_PATH = "/mnt/c/Users/biend/OneDrive/Máy tính/spark-streaming-lab/peft"
WSL_OUTPUT_DIR = "/mnt/c/Users/biend/OneDrive/Máy tính/spark-streaming-lab/output"

REPO_CLONE_URL = "https://github.com/huggingface/peft.git"

# chọn đường dẫn phù hợp với môi trường đang chạy
if platform.system() == "Windows":
    REPO_ROOT = Path(WIN_REPO_PATH)
    OUTPUT_DIR = Path(WIN_OUTPUT_DIR)
else:
    # linux / WSL / macOS
    REPO_ROOT = Path(WSL_REPO_PATH)
    OUTPUT_DIR = Path(WSL_OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)   # tạo thư mục output nếu chưa có

print(f"Python     : {sys.version}")
print(f"Platform   : {platform.system()} – {platform.version()}")
print(f"PROJECT    : {PROJECT_ROOT}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


Python     : 3.11.5 | packaged by conda-forge | (main, Aug 27 2023, 03:23:48) [MSC v.1936 64 bit (AMD64)]
Platform   : Windows – 10.0.19045
PROJECT    : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab
REPO_ROOT  : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\peft
OUTPUT_DIR : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output


### 1. Kiểm tra & Clone Repository

In [2]:
def clone_repo_if_needed(repo_root: Path, clone_url: str) -> None:
    """
    Kiểm tra xem repo đã tồn tại tại `repo_root` chưa.
    - Nếu đã có thư mục `.git`-> bỏ qua, in thông báo.
    - Nếu chưa -> thực hiện `git clone --depth 1` (shallow clone để tiết kiệm băng thông).
    """
    git_dir = repo_root / ".git"

    if git_dir.exists():
        print(f"[INFO] Repository đã tồn tại tại: {repo_root}")
        print(f"[INFO] Bỏ qua bước clone.")
        return

    # đảm bảo thư mục cha tồn tại
    repo_root.parent.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Repo chưa tồn tại. Bắt đầu clone (shallow --depth 1)...")
    print(f"[INFO] URL  : {clone_url}")
    print(f"[INFO] Đích : {repo_root}")

    cmd = [
        "git", "clone",
        "--depth", "1",          # shallow clone: chỉ lấy commit mới nhất
        clone_url,
        str(repo_root)
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    # in stdout & stderr để dễ debug
    if result.stdout:
        print("[git stdout]\n" + result.stdout)
    if result.stderr:
        print("[git stderr]\n" + result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"git clone thất bại với exit code {result.returncode}")

    print(f"[OK] Clone thành công -> {repo_root}")


# chạy
clone_repo_if_needed(REPO_ROOT, REPO_CLONE_URL)

# xác nhận thư mục tồn tại sau khi clone  
assert REPO_ROOT.exists(), f"[ERROR] Không tìm thấy repo tại {REPO_ROOT}"
print(f"\n[OK] Repo hợp lệ – thư mục tồn tại: {REPO_ROOT}")

[INFO] Repository đã tồn tại tại: C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\peft
[INFO] Bỏ qua bước clone.

[OK] Repo hợp lệ – thư mục tồn tại: C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\peft


### 2. Duyệt đệ quy & thu thập tất cả file `.py`

In [3]:
# tìm các file đệ quy
all_py_files_absolute = sorted(REPO_ROOT.rglob("*.py"))

# chuyển sang đường dẫn tương đối để dễ đọc  
all_py_files_relative = [
    p.relative_to(REPO_ROOT)
    for p in all_py_files_absolute
]

total_found = len(all_py_files_relative)
print(f"[OK] Tổng số file .py tìm được: {total_found}")
print("\nVí dụ 10 file đầu tiên:")
for rel_path in all_py_files_relative[:10]:
    print(f"  {rel_path}")

[OK] Tổng số file .py tìm được: 431

Ví dụ 10 file đầu tiên:
  docs\source\_config.py
  examples\adamss_finetuning\glue_adamss_asa_example.py
  examples\adamss_finetuning\glue_adamss_asa_manual_example.py
  examples\adamss_finetuning\image_classification_adamss_asa.py
  examples\adamss_finetuning\test_adamss_quick.py
  examples\alora_finetuning\alora_finetuning.py
  examples\arrow_multitask\arrow_phi3_mini.py
  examples\bdlora_finetuning\chat.py
  examples\beft_finetuning\beft_finetuning.py
  examples\boft_controlnet\__init__.py


### 3. Phân loại file `.py`

**Mục tiêu:** Loại bỏ các file không liên quan trước khi đưa vào CPG Parser, giúp giảm nhiễu
và tăng chất lượng đồ thị phụ thuộc.

**Bốn nhóm phân loại và lý do loại trừ:**

| Nhóm | Tiêu chí | Lý do loại trừ |
|:-----|:---------|:---------------|
| **test** | Thư mục chứa `test`/`tests`; tên file bắt đầu bằng `test_` | Test code không đại diện cho logic nghiệp vụ, chứa mock và fixture gây nhiễu CPG |
| **setup** | Tên file chính xác là `setup.py`, `__init__.py`, `conftest.py` (ngoài thư mục test) | File scaffolding/packaging, không chứa logic domain |
| **auto-generated** | Nằm trong `build/`, `dist/`, `*.egg-info/`; hoặc header chứa `DO NOT EDIT` / `auto-generated` | Code sinh tự động thường bị flatten, không phản ánh cấu trúc thiết kế thực |
| **source** | Tất cả file còn lại | Corpus chính cho CPG Parser Service ở Task 2 |

**Thứ tự ưu tiên phân loại:** `auto-generated` -> `setup` (exact filename) -> `test` (path/prefix) -> `source`

In [4]:
# các hằng số phân loại gồm

# từ khoá xuất hiện trong tên THƯ MỤC → nhóm 'test'
TEST_DIR_KEYWORDS = {"test", "tests"}
# tiền tố tên file -> nhóm 'test' (ví dụ: test_lora.py)
TEST_FILENAME_PREFIX = "test_"

# tên file chính xác -> nhóm 'setup'
# conftest.py đặt ở đây (KHÔNG trong TEST_DIR_KEYWORDS) để tránh bị phân loại sai thành 'test' do 'test' là substring của 'conftest'
SETUP_FILENAMES = {"setup.py", "conftest.py", "__init__.py"}

# thư mục chỉ điểm -> nhóm 'auto-generated'
AUTOGEN_DIRS = {"build", "dist", ".egg-info"}
# marker trong header file -> nhóm 'auto-generated'
AUTOGEN_MARKERS = ("auto-generated", "autogenerated", "do not edit", "do not modify")
# số dòng tối đa đọc để tìm auto-generated marker (tránh đọc file lớn)
HEADER_SCAN_LINES = 10


def is_auto_generated(abs_path: Path) -> bool:
    """
    heuristic phát hiện file auto-generated:
    - Nằm trong thư mục build/, dist/, hoặc *.egg-info/ (kiểm tra theo path parts)
    - Header 10 dòng đầu chứa marker 'DO NOT EDIT' / 'auto-generated'

    nhận abs_path (Path tuyệt đối) để tránh phụ thuộc vào REPO_ROOT global.
    """
    # kiểm tra theo tên thư mục trong đường dẫn
    for part in abs_path.parts:
        if part in AUTOGEN_DIRS or part.endswith(".egg-info"):
            return True

    # kiểm tra nội dung header
    try:
        with open(abs_path, encoding="utf-8", errors="ignore") as fh:
            header_lines = []
            for i, line in enumerate(fh):
                if i >= HEADER_SCAN_LINES:
                    break
                header_lines.append(line.lower())
            header = "".join(header_lines)
        return any(marker in header for marker in AUTOGEN_MARKERS)
    except OSError:
        return False


def classify_file(rel_path: Path) -> str:
    """
    phân loại file python theo 4 nhóm.
    trả về: 'auto-generated' | 'setup' | 'test' | 'source'

    thứ tự ưu tiên phân loại:
      1. auto-generated  – loại trừ trước, ngay cả khi tên file trùng nhóm khác
      2. setup           – khớp CHÍNH XÁC tên file (conftest.py, __init__.py, setup.py)
                           PHẢI kiểm tra TRƯỚC keyword test, vì 'test' là substring
                           của 'conftest' -> nếu đảo thứ tự, conftest.py sẽ bị nhầm
                           thành 'test'
      3. test            – thư mục chứa 'test'/'tests' hoặc tên file bắt đầu 'test_'
      4. source          – tất cả còn lại
    """
    abs_path    = REPO_ROOT / rel_path
    filename    = rel_path.name.lower()
    # Lấy các phần thư mục (bỏ tên file cuối cùng) để kiểm tra keyword
    dir_parts   = [p.lower() for p in rel_path.parts[:-1]]

    # 1. auto-generated  
    if is_auto_generated(abs_path):
        return "auto-generated"

    # 2. setup files
    if filename in SETUP_FILENAMES:
        return "setup"

    # 3. test files 
    if any(kw == part for part in dir_parts for kw in TEST_DIR_KEYWORDS):
        return "test"
    if filename.startswith(TEST_FILENAME_PREFIX):
        return "test"

    # 4. source 
    return "source"


# phân loại
categories = [classify_file(p) for p in all_py_files_relative]

# đếm số lượng từng nhóm
from collections import Counter
category_counts = Counter(categories)

print("=" * 50)
print("KẾT QUẢ PHÂN LOẠI FILE .py")
print("=" * 50)
print(f"  {'test':<18}: {category_counts.get('test', 0):>5} file(s)")
print(f"  {'setup':<18}: {category_counts.get('setup', 0):>5} file(s)")
print(f"  {'auto-generated':<18}: {category_counts.get('auto-generated', 0):>5} file(s)")
print(f"  {'source':<18}: {category_counts.get('source', 0):>5} file(s)")
print("-" * 50)
print(f"  {'TỔNG':<18}: {sum(category_counts.values()):>5} file(s)")

# conftest.py được phân loại đúng là 'setup', không phải 'test'
conftest_files = [(p, c) for p, c in zip(all_py_files_relative, categories)
                  if p.name == "conftest.py"]
print(f"\n── Kiểm tra conftest.py (phải là 'setup', không phải 'test') ────────")
if conftest_files:
    for path, cat in conftest_files[:5]:
        status = "OK" if cat == "setup" else "✗ LỖI"
        print(f"  [{status}] {path}  ->  {cat}")
else:
    print("  (Không tìm thấy conftest.py trong repo)")

KẾT QUẢ PHÂN LOẠI FILE .py
  test              :    63 file(s)
  setup             :    59 file(s)
  auto-generated    :     0 file(s)
  source            :   309 file(s)
--------------------------------------------------
  TỔNG              :   431 file(s)

── Kiểm tra conftest.py (phải là 'setup', không phải 'test') ────────
  [OK] tests\conftest.py  ->  setup


#### 3a. Kiểm chứng heuristic is_auto_generated() (Smoke Test)

Do repository ban đầu không chứa các file sinh tự động nên kết quả phát hiện là 0 file. Để kiểm tra hàm `is_auto_generated()`, nhóm tạo một số file và thư mục giả lập (ví dụ build/, dist/,...) rồi chạy lại chương trình. Nếu các file này được phát hiện đúng là auto-generated thì có thể kết luận detector hoạt động như mong đợi. Sau khi kiểm tra, nhóm xóa các file giả lập để giữ nguyên trạng thái của repository.

In [5]:
# kiểm tra nhanh hàm is_auto_generated()
import tempfile
import textwrap

# trường hợp 1: File có dòng "DO NOT EDIT" -> dự kiến trả về True
with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp:
    tmp.write(textwrap.dedent("""\
        # DO NOT EDIT - This file is auto-generated by the build system.
        # Any manual changes will be overwritten on next build.
        print('hello from generated file')
    """))
    tmp_path = Path(tmp.name)

result_autogen = is_auto_generated(tmp_path)
tmp_path.unlink()   # xóa file sau khi kiểm tra

# trường hợp 2: File Python bình thường -> dự kiến trả về False
with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp2:
    tmp2.write("def my_func():\n    return 42\n")
    tmp2_path = Path(tmp2.name)

result_normal = is_auto_generated(tmp2_path)
tmp2_path.unlink()

# trường hợp 3: Đường dẫn có thư mục build -> dự kiến trả về True
# không cần tạo file thật vì hàm chỉ kiểm tra tên thư mục
if platform.system() == "Windows":
    fake_build_path = Path(r"C:\project\build\generated_module.py")
else:
    fake_build_path = Path("/project/build/generated_module.py")

result_build = is_auto_generated(fake_build_path)

# hiển thị kết quả
print("=" * 55)
print("Kiểm tra hàm is_auto_generated()")
print("=" * 55)

test_cases = [
    ("File có 'DO NOT EDIT'", result_autogen, True),
    ("File Python bình thường", result_normal, False),
    ("Đường dẫn chứa 'build/'", result_build, True),
]

all_passed = True

for desc, got, expected in test_cases:
    if got == expected:
        print(f"[OK] {desc}")
    else:
        print(f"[Lỗi] {desc} (kết quả: {got}, mong đợi: {expected})")
        all_passed = False

assert all_passed, "Có test chưa đúng, cần kiểm tra lại hàm."

print("\nHoàn thành kiểm tra. Tất cả các trường hợp đều đúng.")

Kiểm tra hàm is_auto_generated()
[OK] File có 'DO NOT EDIT'
[OK] File Python bình thường
[OK] Đường dẫn chứa 'build/'

Hoàn thành kiểm tra. Tất cả các trường hợp đều đúng.


### 4. Tạo DataFrame & Lưu ra CSV

In [6]:
def count_lines(abs_path: Path) -> int:
    """Đếm số dòng của file, nếu không đọc được thì trả về -1."""
    try:
        with open(abs_path, encoding="utf-8", errors="ignore") as fh:
            return sum(1 for _ in fh)
    except OSError:
        return -1


# thu thập thông tin của từng file Python
records = []

for rel_path, category in zip(all_py_files_relative, categories):
    abs_path = REPO_ROOT / rel_path

    try:
        size_bytes = abs_path.stat().st_size
    except OSError:
        size_bytes = -1

    records.append({
        "relative_path": str(rel_path),
        "size_bytes": size_bytes,
        "category": category,
        "num_lines": count_lines(abs_path),
    })

# tạo DataFrame
df = pd.DataFrame(
    records,
    columns=["relative_path", "size_bytes", "category", "num_lines"]
)

# lưu kết quả để dùng cho các phần sau
csv_output_path = OUTPUT_DIR / "discovered_files.csv"
df.to_csv(csv_output_path, index=False, encoding="utf-8")

print(f"Đã lưu file CSV: {csv_output_path}")
print(f"Số lượng dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột\n")

# hiển thị thử vài dòng đầu
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)


print("10 dòng đầu của DataFrame")
print("-" * 60)

df.head(10)

Đã lưu file CSV: C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output\discovered_files.csv
Số lượng dữ liệu: 431 dòng, 4 cột

10 dòng đầu của DataFrame
------------------------------------------------------------


,relative_path,size_bytes,category,num_lines
0,docs\source\_config.py,287,source,7
1,examples\adamss_finetuning\glue_adamss_asa_example.py,15224,source,382
2,examples\adamss_finetuning\glue_adamss_asa_manual_example.py,16825,source,411
3,examples\adamss_finetuning\image_classification_adamss_asa.py,17368,source,452
4,examples\adamss_finetuning\test_adamss_quick.py,4587,test,176
5,examples\alora_finetuning\alora_finetuning.py,9839,source,259
6,examples\arrow_multitask\arrow_phi3_mini.py,14755,source,383
7,examples\bdlora_finetuning\chat.py,1972,source,64
8,examples\beft_finetuning\beft_finetuning.py,4083,source,122
9,examples\boft_controlnet\__init__.py,0,setup,0


In [7]:
# Thống kê nhanh theo category
summary_df = (
    df.groupby("category")
      .agg(
          num_files   = ("relative_path", "count"),
          total_bytes = ("size_bytes",    "sum"),
          total_lines = ("num_lines",     "sum"),
          avg_lines   = ("num_lines",     "mean"),
      )
      .reset_index()
      .sort_values("num_files", ascending=False)
)
summary_df["avg_lines"] = summary_df["avg_lines"].round(1)

print("Thống kê theo category")
summary_df

Thống kê theo category


,category,num_files,total_bytes,total_lines,avg_lines
1,source,309,4052642,93538,302.7
2,test,63,2065517,48096,763.4
0,setup,59,75268,2144,36.3


### 5. Bảng tổng kết cuối cùng (Lab Report)

In [8]:
# ── Các con số chính cần ghi vào Lab Report ───────────────────────────────
total_py_files = len(df)
n_test         = category_counts.get("test", 0)
n_setup        = category_counts.get("setup", 0)
n_autogen      = category_counts.get("auto-generated", 0)
n_excluded     = n_test + n_setup + n_autogen      # tổng file bị loại trừ
n_source       = category_counts.get("source", 0)  # file đưa vào Parser

print("╔══════════════════════════════════════════════════════╗")
print("║           TASK 1 – FINAL SUMMARY (Lab Report)        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Repository : huggingface/peft                       ║")
print(f"║  URL        : https://github.com/huggingface/peft    ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Total .py files discovered        : {total_py_files:>5}           ║")
print("╠──────────────────────────────────────────────────────╣")
print(f"║  Excluded – test files             : {n_test:>5}           ║")
print(f"║  Excluded – setup/init files       : {n_setup:>5}           ║")
print(f"║  Excluded – auto-generated files   : {n_autogen:>5}           ║")
print(f"║  Total excluded                    : {n_excluded:>5}           ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  SOURCE files -> CPG Parser Service : {n_source:>5}          ║")
print("╚══════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║           TASK 1 – FINAL SUMMARY (Lab Report)        ║
╠══════════════════════════════════════════════════════╣
║  Repository : huggingface/peft                       ║
║  URL        : https://github.com/huggingface/peft    ║
╠══════════════════════════════════════════════════════╣
║  Total .py files discovered        :   431           ║
╠──────────────────────────────────────────────────────╣
║  Excluded – test files             :    63           ║
║  Excluded – setup/init files       :    59           ║
║  Excluded – auto-generated files   :     0           ║
║  Total excluded                    :   122           ║
╠══════════════════════════════════════════════════════╣
║  SOURCE files -> CPG Parser Service :   309          ║
╚══════════════════════════════════════════════════════╝


### 6. Reflection

**What worked**
- `git clone --depth 1` giúp lấy repo `peft` nhanh và nhẹ, đủ dùng cho việc phân tích tĩnh mã nguồn.
- `pathlib.Path.rglob("*.py")` duyệt đệ quy toàn bộ 431 file `.py` mà không cần xử lý riêng cho từng hệ điều hành.
- Heuristic phân loại 4 nhóm (`auto-generated` -> `setup` -> `test` -> `source`) cho kết quả nhất quán, đã kiểm chứng bằng smoke test riêng cho `is_auto_generated()` (3/3 test case PASS).

**What failed**
- Bản đầu tiên đặt thứ tự ưu tiên là `test` trước `setup`, khiến `conftest.py` bị nhận nhầm thành `test` (do `"test"` là substring của `"conftest"`). Notebook đã bổ sung một bước kiểm tra riêng (in ra danh sách `conftest.py` và category tương ứng) để phát hiện lỗi này.
- Repo `peft` không có sẵn file/thư mục auto-generated (`build/`, `dist/`, `.egg-info/`) nên nhóm không thể xác nhận `is_auto_generated()` hoạt động đúng chỉ bằng cách chạy trên repo thật.

**How resolved**
- Đổi thứ tự ưu tiên phân loại: kiểm tra khớp tên file chính xác (`setup`) trước khi kiểm tra từ khoá `test`, giải quyết dứt điểm lỗi `conftest.py`.
- Viết smoke test tạo file/thư mục giả lập (có marker `DO NOT EDIT`, path chứa `build/`) để kiểm chứng `is_auto_generated()` hoạt động đúng, bù lại việc repo thật không có ca auto-generated nào.

## **Task 2: Incremental CPG Parser Service**

### Mục tiêu
- Xây dựng **Parser Service** (`cpg_parser.py`) xử lý file Python **từng file một** (không parse cả repo một lần).
- Dùng thư viện **`ast`** (Python standard library) để trích xuất:
  - **AST nodes**
  - **CFG edges** (control flow)
  - **DFG edges** (data flow)
  - **CALL edges** (function/method calls)
- Mỗi phần tử có **ID ổn định** (UUIDv5) để reprocess không tạo trùng downstream.
- Emit 3 luồng sự kiện chính: **nodes**, **edges**, **metadata** (+ `errors` khi parse lỗi).
- Service hoạt động với **bounded memory**: parse 1 file → emit → flush → sang file tiếp theo.

### Thiết kế nhanh

| Thành phần | Cách làm |
|:---|:---|
| Parser | `ast.parse` + `ast.NodeVisitor` trong `cpg_parser.py` |
| Stable node ID | `uuid.uuid5(NAMESPACE_DNS, f"{file_path}:{ast_path}")` |
| Stable edge ID | `uuid.uuid5(NAMESPACE_DNS, f"{src}->{tgt}:{edge_type}")` |
| Metadata ID | `uuid.uuid5(NAMESPACE_DNS, file_path)` |
| Input | `output/discovered_files.csv` (chỉ `category == source`) |
| Chế độ chạy | `--dry-run` (không Kafka) / emit Kafka (`--bootstrap-servers`) |


### 0. Kiểm tra đầu vào Task 1 & cấu hình Parser


In [9]:
import json
import hashlib
import uuid
from collections import Counter

# Dùng lại PROJECT_ROOT / REPO_ROOT / OUTPUT_DIR từ cell cấu hình Task 1
assert "PROJECT_ROOT" in dir(), "Hãy chạy cell cấu hình đường dẫn (Task 1) trước."

PARSER_SCRIPT = PROJECT_ROOT / "cpg_parser.py"
DISCOVERED_CSV = OUTPUT_DIR / "discovered_files.csv"
KAFKA_BOOTSTRAP = "localhost:9092"
SCHEMA_VERSION = "1.0.0"
DEMO_LIMIT = 5  # số file dùng cho demo notebook (đủ bằng chứng, không quá lâu)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("REPO_ROOT      :", REPO_ROOT)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("PARSER_SCRIPT  :", PARSER_SCRIPT, "| exists =", PARSER_SCRIPT.exists())
print("DISCOVERED_CSV :", DISCOVERED_CSV, "| exists =", DISCOVERED_CSV.exists())
print("Kafka brokers  :", KAFKA_BOOTSTRAP)
print("Schema version :", SCHEMA_VERSION)
print("Demo limit     :", DEMO_LIMIT)

assert PARSER_SCRIPT.exists(), f"Thiếu {PARSER_SCRIPT}"
assert DISCOVERED_CSV.exists(), f"Thiếu {DISCOVERED_CSV} — chạy Task 1 trước"
assert REPO_ROOT.exists(), f"Thiếu repo tại {REPO_ROOT}"

df_all = pd.read_csv(DISCOVERED_CSV)
df_source = df_all[df_all["category"] == "source"].copy()
print(f"\n[OK] CSV có {len(df_all)} file .py, trong đó source = {len(df_source)}")
print(df_source.head(5).to_string(index=False))


PROJECT_ROOT   : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab
REPO_ROOT      : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\peft
OUTPUT_DIR     : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output
PARSER_SCRIPT  : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\cpg_parser.py | exists = True
DISCOVERED_CSV : C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output\discovered_files.csv | exists = True
Kafka brokers  : localhost:9092
Schema version : 1.0.0
Demo limit     : 5

[OK] CSV có 431 file .py, trong đó source = 309
                                                relative_path  size_bytes category  num_lines
                                       docs\source\_config.py         287   source          7
        examples\adamss_finetuning\glue_adamss_asa_example.py       15224   source        382
 examples\adamss_finetuning\glue_adamss_asa_manual_example.py       16825   source        411
examples\adamss_finetuning\image_classification_adamss_asa.py    

### 1. Dry-run Parser Service (không gửi Kafka)

Chạy `cpg_parser.py --dry-run` trên vài file nguồn đầu tiên để lấy:
- số node/edge từng file
- **sample JSON** cho node / edge / metadata
- xác nhận mỗi message có `schema_version` và `event_time`


In [10]:
#!pip install kafka-python

In [11]:
cmd_dry = [
    sys.executable, str(PARSER_SCRIPT),
    "--dry-run",
    "--limit", str(DEMO_LIMIT),
    "--repo-root", str(REPO_ROOT),
    "--discovered-csv", str(DISCOVERED_CSV),
    "--schema-version", SCHEMA_VERSION,
]
#print("CMD:", " ".join(cmd_dry))
print("-" * 72)

proc = subprocess.run(
    cmd_dry,
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(proc.stdout)
if proc.returncode != 0:
    print("[STDERR]")
    print(proc.stderr)
    raise RuntimeError(f"Dry-run thất bại, exit={proc.returncode}")

log_dry = OUTPUT_DIR / "task2_dryrun.log"
log_dry.write_text(proc.stdout, encoding="utf-8")
print(f"[OK] Đã lưu log -> {log_dry}")


------------------------------------------------------------------------
=== Starting CPG Parser Service ===
Discovered CSV : C:\Users\biend\OneDrive\M�y t�nh\spark-streaming-lab\output\discovered_files.csv
Repo Root      : C:\Users\biend\OneDrive\M�y t�nh\spark-streaming-lab\peft
Kafka Brokers  : localhost:9092
Schema Version : 1.0.0
Dry Run Mode   : True

Found 5 source files to parse.
[1/5] Processing: docs\source\_config.py ... Parsed successfully (Nodes: 5, Edges: 6) [DRY RUN]

--- SAMPLE NODE EVENT ---
{
  "id": "c497a51d-c0ad-5f8e-b20e-c6c29c746e36",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-15T15:27:38.488978Z"
}

--- SAMPLE EDGE EVENT ---
{
  "id": "810eb670-85ff-5a98-b383-c32acdc2f864",
  "source_id": "c497a51d-c0ad-5f8e-b20e-c6c29c746e36",
  "target_id": "909e404

### 2. Phân tích chi tiết 1 file + kiểm chứng Stable ID (idempotency)

Parse cùng một file **hai lần**. Nếu ID ổn định, tập `id` của nodes/edges/metadata lần 1 và lần 2 phải **trùng khớp 100%**.


In [12]:
sys.path.insert(0, str(PROJECT_ROOT))
from cpg_parser import process_file, normalize_rel_path

sample_rel = df_source.iloc[0]["relative_path"]
print("Sample file:", sample_rel)

nodes1, edges1, meta1 = process_file(sample_rel, REPO_ROOT, SCHEMA_VERSION)
nodes2, edges2, meta2 = process_file(sample_rel, REPO_ROOT, SCHEMA_VERSION)

ids_n1 = {n["id"] for n in nodes1}
ids_n2 = {n["id"] for n in nodes2}
ids_e1 = {e["id"] for e in edges1}
ids_e2 = {e["id"] for e in edges2}

print(f"Nodes  run1={len(nodes1)} run2={len(nodes2)} | ID overlap = {len(ids_n1 & ids_n2)}/{len(ids_n1)}")
print(f"Edges  run1={len(edges1)} run2={len(edges2)} | ID overlap = {len(ids_e1 & ids_e2)}/{len(ids_e1)}")
print(f"Metadata id run1={meta1['id']}")
print(f"Metadata id run2={meta2['id']}")
print(f"Metadata ID identical? {meta1['id'] == meta2['id']}")

assert ids_n1 == ids_n2, "Node IDs không ổn định giữa 2 lần parse!"
assert ids_e1 == ids_e2, "Edge IDs không ổn định giữa 2 lần parse!"
assert meta1["id"] == meta2["id"], "Metadata ID không ổn định!"
print("\n[OK] Stable ID verified — reprocess cùng nội dung không tạo ID mới.")

edge_types = Counter(e["type"] for e in edges1)
print("\nEdge type counts:", dict(edge_types))

print("\n--- SAMPLE NODE ---")
print(json.dumps(nodes1[0], indent=2, ensure_ascii=False))
print("\n--- SAMPLE EDGE ---")
print(json.dumps(edges1[0], indent=2, ensure_ascii=False))
print("\n--- SAMPLE METADATA ---")
print(json.dumps(meta1, indent=2, ensure_ascii=False))

for label, obj in [("node", nodes1[0]), ("edge", edges1[0]), ("metadata", meta1)]:
    assert "schema_version" in obj, f"{label} thiếu schema_version"
    assert "event_time" in obj, f"{label} thiếu event_time"
print("\n[OK] schema_version + event_time có trên mọi loại event.")


Sample file: docs\source\_config.py
Nodes  run1=5 run2=5 | ID overlap = 5/5
Edges  run1=6 run2=6 | ID overlap = 6/6
Metadata id run1=fbc8c700-7469-5400-898f-bd8616625351
Metadata id run2=fbc8c700-7469-5400-898f-bd8616625351
Metadata ID identical? True

[OK] Stable ID verified — reprocess cùng nội dung không tạo ID mới.

Edge type counts: {'AST': 4, 'CFG': 2}

--- SAMPLE NODE ---
{
  "id": "c497a51d-c0ad-5f8e-b20e-c6c29c746e36",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-15T15:27:39.263448Z"
}

--- SAMPLE EDGE ---
{
  "id": "810eb670-85ff-5a98-b383-c32acdc2f864",
  "source_id": "c497a51d-c0ad-5f8e-b20e-c6c29c746e36",
  "target_id": "909e4049-6859-5c41-80ed-022598c06caa",
  "type": "AST",
  "schema_version": "1.0.0",
  "event_time": "2026-07-15T15:27:39.263448Z"
}

--- SAMPLE 

### 3. Tổng hợp số liệu dry-run trên DEMO_LIMIT file nguồn

Parse lần lượt `DEMO_LIMIT` file trong notebook để ghi **bảng số liệu** (bằng chứng Task 2).


In [13]:
rows = []
total_nodes = total_edges = 0
edge_counter = Counter()

for rel in df_source["relative_path"].head(DEMO_LIMIT).tolist():
    nodes, edges, meta = process_file(rel, REPO_ROOT, SCHEMA_VERSION)
    et = Counter(e["type"] for e in edges)
    rows.append({
        "file_path": meta["file_path"],
        "num_lines": meta["num_lines"],
        "sha256_short": meta["sha256"][:12],
        "nodes": len(nodes),
        "edges": len(edges),
        "AST": et.get("AST", 0),
        "CFG": et.get("CFG", 0),
        "DFG": et.get("DFG", 0),
        "CALL": et.get("CALL", 0),
    })
    total_nodes += len(nodes)
    total_edges += len(edges)
    edge_counter.update(et)

stats_df = pd.DataFrame(rows)
display(stats_df)

print("=== TASK 2 DRY-RUN SUMMARY ===")
print(f"Files parsed : {len(rows)}")
print(f"Total nodes  : {total_nodes}")
print(f"Total edges  : {total_edges}")
print(f"Edge breakdown: {dict(edge_counter)}")

stats_path = OUTPUT_DIR / "task2_parse_stats.csv"
stats_df.to_csv(stats_path, index=False)
print(f"[OK] Saved stats -> {stats_path}")


,file_path,num_lines,sha256_short,nodes,edges,AST,CFG,DFG,CALL
0,docs/source/_config.py,7,62bc147b19a0,5,6,4,2,0,0
1,examples/adamss_finetuning/glue_adamss_asa_example.py,382,62678c022b20,2222,3931,2221,1447,159,104
2,examples/adamss_finetuning/glue_adamss_asa_manual_example.py,411,b7739cdbdf24,2175,3781,2174,1412,93,102
3,examples/adamss_finetuning/image_classification_adamss_asa.py,452,11f0174e1638,2319,4123,2318,1506,176,123
4,examples/alora_finetuning/alora_finetuning.py,259,4ca81443b4a4,1227,2045,1226,676,97,46


=== TASK 2 DRY-RUN SUMMARY ===
Files parsed : 5
Total nodes  : 7948
Total edges  : 13886
Edge breakdown: {'AST': 7943, 'CFG': 5043, 'CALL': 375, 'DFG': 525}
[OK] Saved stats -> C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output\task2_parse_stats.csv


### 4. Reflection – Task 2

**What worked**
- `ast` đủ để dựng AST parent–child, suy CFG tuần tự trên statement/expression, DFG định nghĩa→dùng, và CALL từ function bao quanh tới `ast.Call`.
- UUIDv5 + đường dẫn POSIX (`as_posix`) giúp ID ổn định giữa Windows/WSL.
- Xử lý từng file + `--limit` / `--file` giúp demo an toàn, tránh OOM.

**What failed / lưu ý**
- CFG/DFG ở đây mang tính demo giáo dục (không phải phân tích liên thủ tục đầy đủ như Joern).
- File syntax lỗi sẽ không tạo nodes — được ghi vào topic `cpg-errors` khi emit Kafka.

**How resolved**
- Chuẩn hoá `file_path` trước khi hash ID; flush Kafka sau mỗi file; thêm `--dry-run` để kiểm chứng trước khi ghi broker.


## **Task 3: Kafka Topic Design**

### Mục tiêu
- Thiết kế **4 Kafka topic** mang event từ Parser Service:
  1. `cpg-nodes` — node events
  2. `cpg-edges` — edge events (AST/CFG/DFG/CALL)
  3. `cpg-metadata` — source metadata events
  4. `cpg-errors` — parser error events
- Mỗi message bắt buộc có **`schema_version`** (forward compatibility) và **`event_time`**.
- Chứng minh Parser (Task 2) **đã ghi thành công** vào broker: list topic, offset/count, sample message.

### Sơ đồ luồng

```
Python source files (.py)
        |
        v
   cpg_parser.py  (incremental, 1 file at a time)
        |
        |---> cpg-nodes      ---> Neo4j Sink (Task 4)
        |---> cpg-edges      ---> Neo4j Sink (Task 4)
        |---> cpg-metadata   ---> Spark -> MongoDB (Task 5)
        +---> cpg-errors     ---> monitoring / debug
```


### 0. Schema thiết kế (message contract)

#### Topic `cpg-nodes`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable node id — **Kafka key** |
| `type` / `label` | string | Loại AST node (`FunctionDef`, `Name`, ...) |
| `file_path` | string | Đường dẫn tương đối (POSIX) |
| `start_line`, `start_column`, `end_line`, `end_column` | int/null | Vị trí nguồn |
| `code` | string | Snippet |
| `properties` | object | name/value phụ |
| `schema_version` | string | vd. `1.0.0` |
| `event_time` | string | ISO-8601 UTC |

#### Topic `cpg-edges`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable edge id — **Kafka key** |
| `source_id`, `target_id` | string | Liên kết tới node ids |
| `type` | string | `AST` / `CFG` / `DFG` / `CALL` |
| `properties` | object (optional) | vd. `callee` cho CALL |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-metadata`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable file id — **Kafka key** |
| `file_path` | string | Đường dẫn tương đối |
| `size_bytes`, `num_lines` | int | Kích thước file |
| `sha256` | string | Hash nội dung |
| `processed_at`, `status` | string | Thời điểm / SUCCESS |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-errors`
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | `uuid5(file_path + ":error")` |
| `file_path`, `error_type`, `error_message`, `stack_trace` | string | Chi tiết lỗi |
| `occurred_at`, `schema_version`, `event_time` | string | Bắt buộc |

**Cấu hình broker (lab):** `replication-factor=1`, `partitions=1` (single-broker WSL).


### 1. Khởi động Kafka (WSL) & tạo 4 topic

Cell dưới gọi script `scripts/setup_kafka.sh` (start ZooKeeper + Broker + create topics nếu chưa có).


In [14]:
SETUP_SCRIPT_WSL = "/mnt/c/Users/biend/OneDrive/Máy tính/spark-streaming-lab/scripts/setup_kafka.sh"
KAFKA_HOME = "/home/pnd/data/kafka_2.12-3.8.0"

def run_wsl(bash_cmd, timeout=180):
    """Chạy lệnh bash trong WSL, trả về CompletedProcess."""
    return subprocess.run(
        ["wsl", "-e", "bash", "-c", bash_cmd],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )

print("[INFO] Chạy setup_kafka.sh ...")
setup_cmd = "sed -i 's/\\r$//' '" + SETUP_SCRIPT_WSL + "' && bash '" + SETUP_SCRIPT_WSL + "'"
r = run_wsl(setup_cmd, timeout=180)
print(r.stdout)
if r.returncode != 0:
    print("[STDERR]", r.stderr)
    raise RuntimeError("setup_kafka.sh thất bại")

print("\n[INFO] kafka-topics --list")
r2 = run_wsl(KAFKA_HOME + "/bin/kafka-topics.sh --list --bootstrap-server localhost:9092")
print(r2.stdout)
topics_found = [t.strip() for t in r2.stdout.splitlines() if t.strip()]
required = {"cpg-nodes", "cpg-edges", "cpg-metadata", "cpg-errors"}
missing = required - set(topics_found)
print("Topics found:", topics_found)
assert not missing, f"Thiếu topic: {missing}"
print("[OK] Đủ 4 topic Task 3 trên broker.")

(OUTPUT_DIR / "task3_topics_list.log").write_text(r2.stdout, encoding="utf-8")


[INFO] Chạy setup_kafka.sh ...
=== Kafka listener config ===
advertised.listeners=PLAINTEXT://localhost:9092
listeners=PLAINTEXT://0.0.0.0:9092
ZooKeeper already listening on 2181
Restarting Kafka broker...
=== Create topics (Task 3) ===
=== Topic list ===
__consumer_offsets
cpg-edges
cpg-errors
cpg-metadata
cpg-nodes
=== Done ===


[INFO] kafka-topics --list
__consumer_offsets
cpg-edges
cpg-errors
cpg-metadata
cpg-nodes

Topics found: ['__consumer_offsets', 'cpg-edges', 'cpg-errors', 'cpg-metadata', 'cpg-nodes']
[OK] Đủ 4 topic Task 3 trên broker.


63

### 2. Describe topics (bằng chứng cấu hình partition / replication)


In [15]:
r3 = run_wsl(KAFKA_HOME + "/bin/kafka-topics.sh --describe --bootstrap-server localhost:9092")
print(r3.stdout)
(OUTPUT_DIR / "task3_topics_describe.log").write_text(r3.stdout, encoding="utf-8")
print(f"[OK] Saved -> {OUTPUT_DIR / 'task3_topics_describe.log'}")


Topic: cpg-errors	TopicId: okMU84cnRmOxDDHkTGMZBg	PartitionCount: 1	ReplicationFactor: 1	Configs: 
	Topic: cpg-errors	Partition: 0	Leader: 0	Replicas: 0	Isr: 0	Elr: N/A	LastKnownElr: N/A
Topic: cpg-metadata	TopicId: SY1T8YeTQxKTO7dk-kvaCQ	PartitionCount: 1	ReplicationFactor: 1	Configs: 
	Topic: cpg-metadata	Partition: 0	Leader: 0	Replicas: 0	Isr: 0	Elr: N/A	LastKnownElr: N/A
Topic: __consumer_offsets	TopicId: MWzt4etISligWMCRDH83iQ	PartitionCount: 50	ReplicationFactor: 1	Configs: compression.type=producer,cleanup.policy=compact,segment.bytes=104857600
	Topic: __consumer_offsets	Partition: 0	Leader: 0	Replicas: 0	Isr: 0	Elr: N/A	LastKnownElr: N/A
	Topic: __consumer_offsets	Partition: 1	Leader: 0	Replicas: 0	Isr: 0	Elr: N/A	LastKnownElr: N/A
	Topic: __consumer_offsets	Partition: 2	Leader: 0	Replicas: 0	Isr: 0	Elr: N/A	LastKnownElr: N/A
	Topic: __consumer_offsets	Partition: 3	Leader: 0	Replicas: 0	Isr: 0	Elr: N/A	LastKnownElr: N/A
	Topic: __consumer_offsets	Partition: 4	Leader: 0	Replicas

### 3. Emit sự kiện từ Parser Service lên Kafka (bằng chứng end-to-end)

Chạy `cpg_parser.py` **không** `--dry-run` với `--limit DEMO_LIMIT` để ghi thật vào 4 topic.


In [16]:
cmd_emit = [
    sys.executable, str(PARSER_SCRIPT),
    "--limit", str(DEMO_LIMIT),
    "--repo-root", str(REPO_ROOT),
    "--discovered-csv", str(DISCOVERED_CSV),
    "--bootstrap-servers", KAFKA_BOOTSTRAP,
    "--schema-version", SCHEMA_VERSION,
]
print("CMD:", " ".join(cmd_emit))
print("-" * 72)

proc_emit = subprocess.run(
    cmd_emit,
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(proc_emit.stdout)
if proc_emit.returncode != 0:
    print("[STDERR]")
    print(proc_emit.stderr)
    raise RuntimeError(f"Emit Kafka thất bại, exit={proc_emit.returncode}")

log_emit = OUTPUT_DIR / "task3_emit.log"
log_emit.write_text(proc_emit.stdout, encoding="utf-8")
print(f"[OK] Đã emit và lưu log -> {log_emit}")


CMD: C:\Users\biend\miniconda3\envs\min_ds-env\python.exe C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\cpg_parser.py --limit 5 --repo-root C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\peft --discovered-csv C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output\discovered_files.csv --bootstrap-servers localhost:9092 --schema-version 1.0.0
------------------------------------------------------------------------
=== Starting CPG Parser Service ===
Discovered CSV : C:\Users\biend\OneDrive\M�y t�nh\spark-streaming-lab\output\discovered_files.csv
Repo Root      : C:\Users\biend\OneDrive\M�y t�nh\spark-streaming-lab\peft
Kafka Brokers  : localhost:9092
Schema Version : 1.0.0
Dry Run Mode   : False

Found 5 source files to parse.
[OK] Connected to Kafka Producer successfully.

[1/5] Processing: docs\source\_config.py ... Parsed and Sent (Nodes: 5, Edges: 6)
[2/5] Processing: examples\adamss_finetuning\glue_adamss_asa_example.py ... Parsed and Sent (Nodes: 2222, Edges: 3931

### 4. Đếm message trên từng topic (offset) — bằng chứng Parser đã ghi


In [17]:
from kafka import KafkaConsumer, TopicPartition

topics = ["cpg-nodes", "cpg-edges", "cpg-metadata", "cpg-errors"]
consumer = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP, request_timeout_ms=15000)

print("=== Kafka topic offsets (bằng chứng ghi thành công) ===")
offset_rows = []
for t in topics:
    tp = TopicPartition(t, 0)
    begin = consumer.beginning_offsets([tp])[tp]
    end = consumer.end_offsets([tp])[tp]
    count = end - begin
    offset_rows.append({"topic": t, "begin": begin, "end": end, "messages": count})
    print(f"  {t:14s}  begin={begin:<8d} end={end:<8d} messages={count}")

consumer.close()

offset_df = pd.DataFrame(offset_rows)
display(offset_df)

assert offset_df.loc[offset_df.topic == "cpg-nodes", "messages"].iloc[0] > 0
assert offset_df.loc[offset_df.topic == "cpg-edges", "messages"].iloc[0] > 0
assert offset_df.loc[offset_df.topic == "cpg-metadata", "messages"].iloc[0] > 0
print("\n[OK] cpg-nodes / cpg-edges / cpg-metadata đều có message.")

offset_df.to_csv(OUTPUT_DIR / "task3_offsets.csv", index=False)
print(f"[OK] Saved -> {OUTPUT_DIR / 'task3_offsets.csv'}")


=== Kafka topic offsets (bằng chứng ghi thành công) ===
  cpg-nodes       begin=0        end=891610   messages=891610
  cpg-edges       begin=0        end=1515980  messages=1515980
  cpg-metadata    begin=0        end=636      messages=636
  cpg-errors      begin=0        end=0        messages=0


,topic,begin,end,messages
0,cpg-nodes,0,891610,891610
1,cpg-edges,0,1515980,1515980
2,cpg-metadata,0,636,636
3,cpg-errors,0,0,0



[OK] cpg-nodes / cpg-edges / cpg-metadata đều có message.
[OK] Saved -> C:\Users\biend\OneDrive\Máy tính\spark-streaming-lab\output\task3_offsets.csv


### 5. Đọc sample message từ Kafka (nodes / edges / metadata)

Chứng minh payload trên broker đúng schema đã thiết kế.


In [18]:
def read_one(topic_name, timeout_ms=8000):
    c = KafkaConsumer(
        topic_name,
        bootstrap_servers=KAFKA_BOOTSTRAP,
        auto_offset_reset="earliest",
        consumer_timeout_ms=timeout_ms,
        value_deserializer=lambda b: json.loads(b.decode("utf-8")),
        key_deserializer=lambda b: b.decode("utf-8") if b else None,
    )
    for msg in c:
        c.close()
        return msg.key, msg.value
    c.close()
    return None, None

samples = {}
for topic in ["cpg-nodes", "cpg-edges", "cpg-metadata"]:
    key, value = read_one(topic)
    samples[topic] = {"key": key, "value": value}
    print(f"\n===== SAMPLE from {topic} =====")
    print("Kafka key:", key)
    print(json.dumps(value, indent=2, ensure_ascii=False) if value else "(no message)")
    if value:
        assert "schema_version" in value, f"{topic} thiếu schema_version"
        assert "event_time" in value, f"{topic} thiếu event_time"

sample_path = OUTPUT_DIR / "task3_kafka_samples.json"
sample_path.write_text(json.dumps(samples, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n[OK] Saved samples -> {sample_path}")


C:\Users\biend\AppData\Local\Temp\ipykernel_2564\639813940.py:2: DeprecationWarning: key_deserializer does not implement kafka.serializer.Deserializer
  c = KafkaConsumer(
C:\Users\biend\AppData\Local\Temp\ipykernel_2564\639813940.py:2: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  c = KafkaConsumer(



===== SAMPLE from cpg-nodes =====
Kafka key: c497a51d-c0ad-5f8e-b20e-c6c29c746e36
{
  "id": "c497a51d-c0ad-5f8e-b20e-c6c29c746e36",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-15T13:50:44.819757Z"
}

===== SAMPLE from cpg-edges =====
Kafka key: 810eb670-85ff-5a98-b383-c32acdc2f864
{
  "id": "810eb670-85ff-5a98-b383-c32acdc2f864",
  "source_id": "c497a51d-c0ad-5f8e-b20e-c6c29c746e36",
  "target_id": "909e4049-6859-5c41-80ed-022598c06caa",
  "type": "AST",
  "schema_version": "1.0.0",
  "event_time": "2026-07-15T13:50:44.819757Z"
}

===== SAMPLE from cpg-metadata =====
Kafka key: fbc8c700-7469-5400-898f-bd8616625351
{
  "id": "fbc8c700-7469-5400-898f-bd8616625351",
  "file_path": "docs/source/_config.py",
  "size_bytes": 287,
  "sha256": "62bc147b19a03f310e079c3f03ada33aaf2d4f

### 6. Tóm tắt bằng chứng Task 2 & Task 3 + Reflection


In [25]:
print("╔══════════════════════════════════════════════════════════╗")
print("║        TASK 2 & 3 – EVIDENCE SUMMARY (Lab Report)        ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Parser script     : cpg_parser.py                       ║")
print(f"║  AST library       : Python ast                          ║")
print(f"║  Schema version    : {SCHEMA_VERSION:<39}║")
print(f"║  Demo files parsed : {DEMO_LIMIT:<39}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Kafka topics                                            ║")
for _, row in offset_df.iterrows():
    print(f"║    {row['topic']:<14} messages = {int(row['messages']):<28}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Artifacts in output/                                    ║")
print("║    task2_dryrun.log / task2_parse_stats.csv              ║")
print("║    task3_topics_list.log / task3_topics_describe.log     ║")
print("║    task3_emit.log / task3_offsets.csv                    ║")
print("║    task3_kafka_samples.json                              ║")
print("╚══════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════╗
║        TASK 2 & 3 – EVIDENCE SUMMARY (Lab Report)        ║
╠══════════════════════════════════════════════════════════╣
║  Parser script     : cpg_parser.py                       ║
║  AST library       : Python ast                          ║
║  Schema version    : 1.0.0                                  ║
║  Demo files parsed : 5                                      ║
╠══════════════════════════════════════════════════════════╣
║  Kafka topics                                            ║
║    cpg-nodes      messages = 891610                      ║
║    cpg-edges      messages = 1515980                     ║
║    cpg-metadata   messages = 636                         ║
║    cpg-errors     messages = 0                           ║
╠══════════════════════════════════════════════════════════╣
║  Artifacts in output/                                    ║
║    task2_dryrun.log / task2_parse_stats.csv              ║
║    task3_topics_

### 7. Reflection – Task 3

**What worked**
- 4 topic tách rõ theo loại event → Neo4j chỉ cần subscribe nodes/edges, Spark/Mongo chỉ cần metadata.
- `schema_version` + `event_time` trên mọi message; Kafka key = stable id hỗ trợ sink idempotent ở Task 4.
- `advertised.listeners=PLAINTEXT://localhost:9092` giúp producer Windows nói chuyện được với broker trong WSL.

**What failed**
- Lần đầu broker advertise hostname WSL (`DESKTOP-....localdomain`) khiến client Windows kết nối metadata fail.

**How resolved**
- Sửa `server.properties` (`listeners=0.0.0.0:9092`, `advertised.listeners=localhost:9092`) và đóng gói vào `scripts/setup_kafka.sh`.